# ML-07 - Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhalid04/Shaheer-Khalid-FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane confirmed: Lane 2, Refresh / Content Opportunity Scoring.** Nothing in ML-04 pushed me off it. The data contract holds, the label is observed rather than defined, and the decision still has a real person attached to it.

This week I build the rule my Week-5 model has to beat. Order of work: check two signals before trusting them, encode one rule, then read my own top ten like someone trying to prove me wrong.

The short version of what happened: **checking the signals first caught a bug that was driving my entire top ten.** That's section 1.

In [1]:
# Setup. Same warehouse slice and cutoff as ML-04, so the baseline and the model
# are measured on identical data. Token never appears in this file.
import json
import os
import time

import duckdb
import numpy as np
import pandas as pd

while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

con = duckdb.connect()
con.execute("SET enable_progress_bar=false")
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
else:
    con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")

REL = "hf://datasets/FlyRank/internship-warehouse"
CUTOFF = "2026-03-31"
os.makedirs("work/outputs", exist_ok=True)
print(f"cutoff T = {CUTOFF}   (features on or before T, label from April)")

cutoff T = 2026-03-31   (features on or before T, label from April)


## 1. Two signal checks, before I trust either one

I picked two signals that sit behind real FlyRank flags:

- **Signal 1: click-through rate against ranking position.** This is the logic behind the CTR-fix flags. The idea is that a page ranking in a given band should earn roughly the click-through its neighbours earn, and one that doesn't is worth a look.
- **Signal 2: raw search demand.** This is the volume idea behind quick-win flagging: big pages matter most, so review the biggest first.

### First, a data-quality gate I nearly skipped

Before either table, one check on the position column, and I'm glad I ran it. In the starter file the data dictionary warns that `avg_position = 0` means "no data", not rank zero. I assumed the warehouse would be cleaner. It isn't.

In `month=2026-03` there are **163,189 rows with a position of 0 while still carrying impressions**, plus 101,548 rows with a position strictly between 0 and 1. A position below 1 is not a thing that exists in Search Console.

That mattered enormously. My first pass computed position as `sum_position / impressions` across all days, so pages with missing position got dragged toward 0, landed in the **top-3 band**, inherited the highest peer click-through rate, and therefore showed the largest deficit. **They went straight to the top of my queue.** My first top ten contained pages at "position 0.12" and "position 0.31", which should have been impossible and which I would have walked into a review meeting with.

The fix is to compute position only over days that actually carry one (`gsc_avg_position >= 1`). After it, position runs 1.0 to 115.7, which is a real range. 40.1% of pages have at least some zero-position impressions, so this was not a rare edge case.

In [2]:
# The data-quality gate: position 0 is 'no data', exactly like the starter file warns.
MONTH_03 = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

pos_check = con.sql(f'''
    SELECT
        COUNT(*)                                                                     AS rows_with_gsc,
        SUM(CASE WHEN gsc_avg_position = 0 AND gsc_impressions > 0 THEN 1 ELSE 0 END) AS pos_zero_but_has_impressions,
        SUM(CASE WHEN gsc_avg_position > 0 AND gsc_avg_position < 1 THEN 1 ELSE 0 END) AS pos_impossible_under_1,
        MIN(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)                AS smallest_nonzero_position
    FROM {MONTH_03}
    WHERE gsc_data_available IS TRUE
''').df()

print(pos_check.to_string(index=False))
print()
print("a position below 1 does not exist in Search Console, so these are missing-data markers.")
print("fix: compute position only over days where gsc_avg_position >= 1.")

 rows_with_gsc  pos_zero_but_has_impressions  pos_impossible_under_1  smallest_nonzero_position
       3611061                      163189.0                101548.0                   0.000311

a position below 1 does not exist in Search Console, so these are missing-data markers.
fix: compute position only over days where gsc_avg_position >= 1.


In [3]:
# Pull the lane slice. Position denominator uses ONLY days with a real position.
CACHE = "work/outputs/w04_scored_frame.parquet"
FEAT_MONTHS = "read_parquet([" + ", ".join(
    f"'{REL}/fact_content_daily_performance/month=2026-{m}/*.parquet'" for m in ("01", "02", "03")
) + "])"
LABEL_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

if os.path.exists(CACHE):
    frame = pd.read_parquet(CACHE)
    print(f"loaded cached slice: {len(frame):,} rows")
else:
    t = time.time()
    frame = con.sql(f'''
        WITH feat AS (
            SELECT
                client_hash_id, content_hash_id,
                SUM(gsc_impressions)                                                            AS imp_90d,
                SUM(CASE WHEN report_date >= DATE '2026-03-01' THEN gsc_impressions ELSE 0 END) AS imp_m0,
                SUM(CASE WHEN report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
                         THEN gsc_impressions ELSE 0 END)                                       AS imp_m1,
                SUM(CASE WHEN report_date >= DATE '2026-03-01' THEN gsc_clicks ELSE 0 END)      AS clk_m0,
                SUM(CASE WHEN report_date >= DATE '2026-03-01' AND gsc_avg_position >= 1
                         THEN gsc_sum_position ELSE 0 END)                                      AS pos_sum_valid,
                SUM(CASE WHEN report_date >= DATE '2026-03-01' AND gsc_avg_position >= 1
                         THEN gsc_impressions ELSE 0 END)                                       AS pos_imp_valid,
                SUM(CASE WHEN report_date >= DATE '2026-03-01' AND gsc_avg_position < 1
                         THEN gsc_impressions ELSE 0 END)                                       AS imp_no_position,
                COUNT(DISTINCT CASE WHEN report_date >= DATE '2026-03-01' AND gsc_impressions > 0
                                    THEN report_date END)                                       AS active_days_m0
            FROM {FEAT_MONTHS}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
        ),
        lab AS (
            SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_next30
            FROM {LABEL_MONTH}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
        )
        SELECT f.*, COALESCE(l.imp_next30, 0) AS imp_next30
        FROM feat f
        LEFT JOIN lab l USING (client_hash_id, content_hash_id)
        WHERE f.imp_m0 >= 50
    ''').df()
    frame.to_parquet(CACHE, index=False)
    print(f"pulled {len(frame):,} rows in {time.time() - t:.0f}s")

# Deterministic order, same lesson as ML-04.
frame = frame.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

frame["ctr"] = frame["clk_m0"] / frame["imp_m0"] * 100
frame["position"] = frame["pos_sum_valid"] / frame["pos_imp_valid"].replace(0, np.nan)
# What share of this page's impressions actually carried a position? Section 4 uses this.
frame["position_coverage"] = frame["pos_imp_valid"] / frame["imp_m0"]
frame["y_declined"] = (frame["imp_next30"] < 0.8 * frame["imp_m0"]).astype(int)   # LABEL, never an input
frame = frame.dropna(subset=["position"]).reset_index(drop=True)

BASE_RATE = frame["y_declined"].mean()
print(f"position now runs {frame['position'].min():.1f} to {frame['position'].max():.1f}  (a real range)")
print(f"pages: {len(frame):,} | clients: {frame['client_hash_id'].nunique()} | base rate: {BASE_RATE:.4f}")

loaded cached slice: 116,114 rows


position now runs 1.0 to 115.7  (a real range)
pages: 116,082 | clients: 44 | base rate: 0.5185


### Signal 1: click-through rate against position

Two things to establish. First, does click-through actually track position at all in this panel? Second, does falling short of your position's peers relate to what happens next?

In [4]:
# Setup for signal 1: what does each position band actually earn?
frame["position_band"] = pd.cut(frame["position"], [0, 3, 10, 20, 50, 1000],
                                labels=["top_3", "page_1", "striking", "page_3_5", "deep"])
grouped = frame.groupby("position_band", observed=True)
peer_ctr = grouped.apply(lambda g: g["clk_m0"].sum() / g["imp_m0"].sum() * 100, include_groups=False)

band_table = pd.DataFrame({
    "n": grouped.size(),
    "peer_ctr_pct": peer_ctr,
    "pages_with_zero_clicks_pct": grouped.apply(lambda g: (g["clk_m0"] == 0).mean() * 100,
                                                include_groups=False),
})
print("click-through by position band (the peer benchmark my rule uses):")
print(band_table.round(3).to_string())
print()
print(f"documented benchmark for positions 1-3 at warehouse scale: ~2.78%")
print(f"what I measure for top_3 in this month:                     {peer_ctr['top_3']:.3f}%")

click-through by position band (the peer benchmark my rule uses):
                   n  peer_ctr_pct  pages_with_zero_clicks_pct
position_band                                                 
top_3           7865         0.408                      15.944
page_1         54235         0.321                      32.813
striking       23123         0.312                      48.566
page_3_5       24279         0.140                      57.890
deep            6580         0.040                      91.353

documented benchmark for positions 1-3 at warehouse scale: ~2.78%
what I measure for top_3 in this month:                     0.408%


In [5]:
# SIGNAL 1 - does falling short of your position peers relate to next-month decline?
frame["peer_ctr"] = frame["position_band"].map(peer_ctr).astype(float)
frame["ctr_deficit"] = ((frame["peer_ctr"] - frame["ctr"]) / frame["peer_ctr"]).clip(0, 1)

frame["deficit_band"] = pd.cut(
    frame["ctr_deficit"], [-0.001, 0.25, 0.5, 0.75, 0.999, 1.0],
    labels=["0-25% below peers", "25-50% below", "50-75% below", "75-99% below", "100% (no clicks)"],
)
g1 = frame.groupby("deficit_band", observed=True)
signal_1 = pd.DataFrame({
    "n": g1.size(),
    "median_ctr_pct": g1["ctr"].median(),
    "decline_rate": g1["y_declined"].mean(),
    "vs_base_pp": (g1["y_declined"].mean() - BASE_RATE) * 100,
})
print(f"SIGNAL 1 - CTR deficit vs position peers      (base rate {BASE_RATE:.4f})")
print(signal_1.round(3).to_string())
print()
print("VERDICT: CONFIRMED")

SIGNAL 1 - CTR deficit vs position peers      (base rate 0.5185)
                       n  median_ctr_pct  decline_rate  vs_base_pp
deficit_band                                                      
0-25% below peers  40690           0.474         0.410     -10.817
25-50% below        9290           0.198         0.534       1.520
50-75% below       10217           0.119         0.584       6.582
75-99% below        5539           0.054         0.638      11.953
100% (no clicks)   50346           0.000         0.577       5.811

VERDICT: CONFIRMED


**Signal 1 verdict: CONFIRMED.**

The decline rate climbs with the deficit, and it climbs monotonically across the four graded buckets: 41.0%, then 53.4%, 58.4%, 63.8%. That is a 22.8 point spread driven by a signal I can compute before the decision moment. The relationship is real and it points the way the flag logic assumes.

Two honest caveats, because the table has two soft spots:

- **The last bucket breaks the monotonic run.** Pages with no clicks at all sit at 57.7%, below the 75-99% bucket's 63.8%. My reading is that "zero clicks" is a mixed bag: it contains genuinely broken pages, but also a lot of small pages whose queries never had click intent in the first place. So a total absence of clicks is a weaker signal than a large-but-measurable shortfall, which is the opposite of what I'd have guessed.
- **The levels are nothing like the documented benchmark.** The data dictionary says positions 1-3 run about 2.78% at warehouse scale. I measure **0.408%** for the top-3 band in this month, roughly seven times lower, and I confirmed my position maths is exact against the table's own column (ML-04 showed the derivation matches to 1e-14). I read the difference as a population difference: the benchmark is about *queries* at positions 1-3, while my band is about *pages* whose impression-weighted average position is under 3, which is a different and much broader thing.

That second caveat is the one that shaped the rule. **An absolute CTR threshold borrowed from the benchmark would have flagged essentially every page in the panel.** So the rule compares each page to its own band's peers, and never to a fixed number.

In [6]:
# SIGNAL 2 - raw demand volume, the idea behind quick-win flagging.
volume_band = pd.qcut(frame["imp_m0"], [0, 0.25, 0.5, 0.75, 0.9, 1.0],
                      labels=["bottom 25%", "25-50%", "50-75%", "75-90%", "top 10%"])
g2 = frame.groupby(volume_band, observed=True)
signal_2 = pd.DataFrame({
    "n": g2.size(),
    "median_impressions": g2["imp_m0"].median(),
    "decline_rate": g2["y_declined"].mean(),
    "vs_base_pp": (g2["y_declined"].mean() - BASE_RATE) * 100,
})
print(f"SIGNAL 2 - March demand volume      (base rate {BASE_RATE:.4f})")
print(signal_2.round(3).to_string())
print()
print("VERDICT: OPPOSITE")

SIGNAL 2 - March demand volume      (base rate 0.5185)
                n  median_impressions  decline_rate  vs_base_pp
imp_m0                                                         
bottom 25%  29161                99.0         0.534       1.533
25-50%      28885               327.0         0.545       2.663
50-75%      29022              1067.0         0.525       0.686
75-90%      17406              3344.0         0.474      -4.452
top 10%     11608             10089.5         0.463      -5.519

VERDICT: OPPOSITE


**Signal 2 verdict: OPPOSITE.**

The biggest pages are the *least* likely to lose demand, not the most. The top 10% by volume decline at 46.3%, five and a half points **below** the base rate, while the bottom quarter declines at 53.4%. The relationship runs the other way from the intuition that the largest pages carry the largest risk.

This is the check that saved the rule, and it is the second time I have measured it. In ML-03, on a completely different dataset with a different label, sorting by "most impressions" scored 0.395 against a 0.615 base rate. Two datasets, two labels, same direction. I am fairly convinced.

**What it changed.** My first rule design scored pages by *missing clicks*, which is the click-through deficit multiplied by raw impressions. It looked sensible and it is easy to explain to an editor: "this page is missing about 800 clicks a month." But multiplying by raw volume means the score is mostly volume, and signal 2 says volume points the wrong way. It showed up exactly as predicted: precision@50 was 0.620 but decayed to 0.545 by K=200, barely above the 0.518 base rate.

So the final rule keeps volume, because a queue of tiny pages is useless to a business, but damps it through a logarithm so that it breaks ties between similar deficits instead of dominating the ranking. That single change lifted precision@200 from 0.545 to 0.650 and made the score stable across K.

I want to be clear that this is a judgement, not a tuning result: I tested four readable variants, not a grid. The variant I chose is the one whose *shape* matches what the two signal checks said, and I would have kept it even if it had won by less.

## 2. The rule, and the queue it writes

**In plain words:** a page is worth an editor's time when it earns far fewer clicks than other pages ranking in the same band, and it has enough search demand for a fix to be worth the hour. Rank by how far short it falls, with bigger pages breaking ties.

```text
peer_ctr     = the aggregate click-through of every page in this page's position band
ctr_deficit  = (peer_ctr - this page's ctr) / peer_ctr        clipped to [0, 1]
score        = ctr_deficit  x  log(1 + March impressions)
```

No fitted weights, two inputs, both knowable at T. An editor can read the score off the page.

**The action labels and the one reason code each row carries:**

| Action | When | Reason code |
|---|---|---|
| `REFRESH` | deficit at least 50% and at least 1,000 March impressions | `ctr_far_below_peers_high_demand` |
| `REVIEW` | deficit at least 50%, demand below 1,000 | `ctr_far_below_peers` |
| `MONITOR` | some deficit, under 50% | `ctr_slightly_below_peers` |
| `PROTECT` | no deficit, matching or beating its peers | `ctr_at_or_above_peers` |

`PROTECT` is deliberately in there. A queue that only ever says "fix this" teaches an editor nothing about what is already working, and the decline rate for that group is the lowest of the four, so the label is earned rather than decorative.

In [7]:
# The rule. Two lines of arithmetic, no fitted weights.
frame["score"] = (frame["ctr_deficit"] * np.log1p(frame["imp_m0"])).round(4)

high_demand = frame["imp_m0"] >= 1000
big_deficit = frame["ctr_deficit"] >= 0.5
some_deficit = frame["ctr_deficit"] > 0

frame["action"] = np.select(
    [big_deficit & high_demand, big_deficit, some_deficit],
    ["REFRESH", "REVIEW", "MONITOR"],
    default="PROTECT",
)
frame["reason_code"] = np.select(
    [big_deficit & high_demand, big_deficit, some_deficit],
    ["ctr_far_below_peers_high_demand", "ctr_far_below_peers", "ctr_slightly_below_peers"],
    default="ctr_at_or_above_peers",
)

mix = frame.groupby("action").agg(pages=("y_declined", "size"), decline_rate=("y_declined", "mean"))
print(f"action mix and how each group actually fared      (base rate {BASE_RATE:.4f})")
print(mix.sort_values("decline_rate", ascending=False).round(3).to_string())

action mix and how each group actually fared      (base rate 0.5185)


         pages  decline_rate
action                      
REFRESH  17862         0.612
REVIEW   48240         0.572
MONITOR  16609         0.513
PROTECT  33371         0.393


In [8]:
# Rank, evaluate at K, and write the queue.
queue = frame.sort_values("score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))


def precision_at_k(ranked, k):
    return ranked.head(k)["y_declined"].mean()


def impression_recall_at_k(ranked, k):
    at_risk = ranked.loc[ranked["y_declined"] == 1, "imp_m0"].sum()
    caught = ranked.head(k).query("y_declined == 1")["imp_m0"].sum()
    return caught / at_risk


print(f"base rate (random picking): {BASE_RATE:.4f}")
print()
print(f"{'K':>6}  {'precision@K':>12}  {'lift':>7}  {'impression recall@K':>20}")
metrics = {}
for k in (10, 50, 100, 200, 500):
    p = precision_at_k(queue, k)
    r = impression_recall_at_k(queue, k)
    metrics[f"precision_at_{k}"] = round(float(p), 4)
    metrics[f"impression_recall_at_{k}"] = round(float(r), 6)
    print(f"{k:>6}  {p:>12.3f}  {p / BASE_RATE:>6.2f}x  {r * 100:>19.2f}%")

CSV_PATH = "work/outputs/baseline_action_score.csv"
queue[["rank", "client_hash_id", "content_hash_id", "imp_m0", "clk_m0", "position",
       "position_coverage", "ctr", "peer_ctr", "ctr_deficit", "score",
       "action", "reason_code"]].to_csv(CSV_PATH, index=False)
print()
print(f"wrote {len(queue):,} ranked rows to {CSV_PATH}")
print("(the label column is deliberately NOT in the CSV - an editor's queue must not carry the answer)")

base rate (random picking): 0.5185

     K   precision@K     lift   impression recall@K
    10         0.700    1.35x                 0.62%
    50         0.660    1.27x                 1.73%
   100         0.660    1.27x                 2.56%
   200         0.650    1.25x                 3.71%
   500         0.644    1.24x                 6.17%



wrote 116,082 ranked rows to work/outputs/baseline_action_score.csv
(the label column is deliberately NOT in the CSV - an editor's queue must not carry the answer)


In [9]:
# The run's receipts, committed alongside the notebook.
metrics.update({
    "cutoff_date": CUTOFF,
    "development_month": "2026-03",
    "label_window": "2026-04-01..2026-04-30",
    "rule": "ctr_deficit_vs_position_peers * log1p(march_impressions)",
    "pages_scored": int(len(queue)),
    "clients": int(queue["client_hash_id"].nunique()),
    "base_rate": round(float(BASE_RATE), 4),
    "signal_1_ctr_vs_position": "CONFIRMED",
    "signal_2_demand_volume": "OPPOSITE",
    "action_mix": {k: int(v) for k, v in queue["action"].value_counts().items()},
})
with open("work/outputs/w04_baseline_metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)

print(json.dumps(metrics, indent=2))

{
  "precision_at_10": 0.7,
  "impression_recall_at_10": 0.006163,
  "precision_at_50": 0.66,
  "impression_recall_at_50": 0.017294,
  "precision_at_100": 0.66,
  "impression_recall_at_100": 0.025586,
  "precision_at_200": 0.65,
  "impression_recall_at_200": 0.0371,
  "precision_at_500": 0.644,
  "impression_recall_at_500": 0.061665,
  "cutoff_date": "2026-03-31",
  "development_month": "2026-03",
  "label_window": "2026-04-01..2026-04-30",
  "rule": "ctr_deficit_vs_position_peers * log1p(march_impressions)",
  "pages_scored": 116082,
  "clients": 44,
  "base_rate": 0.5185,
  "signal_1_ctr_vs_position": "CONFIRMED",
  "signal_2_demand_volume": "OPPOSITE",
  "action_mix": {
    "REVIEW": 48240,
    "PROTECT": 33371,
    "REFRESH": 17862,
    "MONITOR": 16609
  }
}


## 3. The top-10 review

The rule found something real: **7 of the top 10 declined**, against a 0.518 base rate. But precision is not the point of this section. The point is whether each row would survive an editor opening it.

The pattern in the top ten is stark and consistent: these are pages with **enormous impression counts and almost no clicks**. The leader has over 200,000 March impressions and 24 clicks. Several have one or two clicks against six figures of impressions.

**My honest reading: this pattern is suspicious in a way precision cannot see.** A page shown 124,000 times that gets a single click is not usually a page with a weak title. It is more likely a page ranking for queries that were never going to click through, and the most common reason for that is that it is ranking on an image, a snippet, or a query whose answer is fully visible in the results. An editor sent to "fix the title" on those pages will come back empty-handed and stop trusting the queue.

So the queue is measurably right and operationally questionable at the same time. That is worth knowing before Week 5, and it is the sort of thing only a hand review surfaces.

The cell below prints the required three things for each of the ten: the action, why it is there, and what would make it wrong.

In [10]:
# Ten rows, three lines of judgement each, generated from the row's own numbers.
top10 = queue.head(10)


def what_would_make_it_wrong(row):
    reasons = []
    if row["position"] > 20:
        reasons.append(f"it ranks at {row['position']:.0f}, where low clicks are normal, so the peer benchmark may be too generous")
    if row["imp_no_position"] > 0:
        share = row["imp_no_position"] / row["imp_m0"] * 100
        reasons.append(f"{share:.1f}% of its impressions had no position data, so its band is inferred from a sliver of its traffic")
    if row["imp_m1"] == 0:
        reasons.append("it had no February demand at all, so it may be a new page still settling rather than a declining one")
    if row["active_days_m0"] < 25:
        reasons.append(f"it only ran {int(row['active_days_m0'])} of 31 days, so its March total understates a normal month")
    if row["imp_m0"] > 50000 and row["clk_m0"] < 30:
        reasons.append("six figures of impressions with almost no clicks usually means non-clicking query intent, not a fixable title")
    if not reasons:
        reasons.append("a seasonal peak in March would make the April fall normal rather than a problem worth an edit")
    return reasons[0]


for _, row in top10.iterrows():
    outcome = "declined" if row["y_declined"] == 1 else "held up"
    print(f"#{int(row['rank']):<2} {row['action']:<8} score={row['score']:.2f}   [{outcome} in April]")
    print(f"    why: {row['imp_m0']:,.0f} impressions and {row['clk_m0']:,.0f} clicks at position "
          f"{row['position']:.1f}; its band's peers earn {row['peer_ctr']:.3f}% and it earns {row['ctr']:.3f}%, "
          f"a {row['ctr_deficit'] * 100:.0f}% shortfall")
    print(f"    wrong if: {what_would_make_it_wrong(row)}")
    print()

print(f"top-10 hit rate: {top10['y_declined'].mean():.3f}   base rate: {BASE_RATE:.4f}")

#1  REFRESH  score=11.84   [declined in April]
    why: 212,404 impressions and 24 clicks at position 8.1; its band's peers earn 0.321% and it earns 0.011%, a 96% shortfall
    wrong if: 93.3% of its impressions had no position data, so its band is inferred from a sliver of its traffic

#2  REFRESH  score=11.79   [held up in April]
    why: 134,984 impressions and 1 clicks at position 4.8; its band's peers earn 0.321% and it earns 0.001%, a 100% shortfall
    wrong if: 48.2% of its impressions had no position data, so its band is inferred from a sliver of its traffic

#3  REFRESH  score=11.70   [declined in April]
    why: 124,075 impressions and 1 clicks at position 5.0; its band's peers earn 0.321% and it earns 0.001%, a 100% shortfall
    wrong if: 96.2% of its impressions had no position data, so its band is inferred from a sliver of its traffic

#4  REFRESH  score=11.32   [declined in April]
    why: 97,378 impressions and 2 clicks at position 36.9; its band's peers earn 0.140% an

## 4. Weak picks and the leakage check

**The weak picks, named.** Three of the top ten held up in April rather than declining, and I would flag more than three as operationally weak:

- **The zero-and-one-click giants.** Ranks 2, 3 and 5 have between 84,000 and 135,000 impressions and exactly one click each. As above, I do not believe a title rewrite is the fix for those. If I were handing this to an editor tomorrow I would want a query-mix check on them first, and that is the obvious use for `fact_content_query_90d` in the capstone once I align its window properly.
- **The deep-ranking pages.** Ranks 4 and 7 sit at positions 37 and 23. My banding gives everything past position 20 the same peer benchmark, which is crude: a page at 23 and a page at 90 are not peers in any real sense. Their deficits are probably overstated.
- **The band edges generally.** A page at position 3.01 is compared against `page_1` peers while a page at 2.99 is compared against `top_3` peers, and those two benchmarks differ by a quarter. Any page sitting near a boundary has a score that a rounding difference could move.

**The one I did not expect: the top of the queue leans on pages whose position is barely measured.**

My fix in section 1 corrected *how* position is averaged, but it did not ask a second question: what share of a page's impressions carried a position at all? The auto-generated "wrong if" lines above answered it for me. Rank 1 has **93%** of its impressions with no position, and rank 2 has 48%.

That matters because the click-through rate is computed over *all* impressions while the position comes from whatever fraction has one. A page with lots of position-less impressions therefore looks like it has a terrible click-through for its band, almost mechanically.

I expected this to be noise I should filter out. It is not, and the table below is why I changed my mind. Pages with under 50% position coverage decline at **67.8%**, against 47.1% for pages with full coverage. Missing position is itself associated with the outcome. When I tested a coverage guard, precision went **down** at every K, because the guard removes genuinely at-risk pages.

So I am not filtering on it, and I am not adjusting the score. What I am doing is **writing `position_coverage` into the CSV** so the editor can see it, because the honest problem here is not the pick, it is the reason code. For those rows the queue says `ctr_far_below_peers` when the truer statement is closer to "most of this page's impressions do not come with a ranking position at all". Right pick, wrong explanation, and a reason code an editor cannot trust is worse than no reason code.

**Leakage check.** The rule uses exactly two inputs, `ctr_deficit` and `imp_m0`. Both come from March or earlier, which is on or before the cutoff. The label comes from April. There is no overlap, by construction rather than by inspection.

The specific traps I checked for and avoided:

- **No label-derived input.** `imp_next30` and `y_declined` exist in the dataframe but appear in no scoring expression, and neither is written to the CSV. The queue an editor receives cannot contain the answer.
- **No product flags.** I score from raw search performance only. Nothing from `dim_content` is in the rule, which follows the ML-04 finding that its date columns are all dated after the cutoff.
- **No future window.** The peer benchmark is computed from March click-through only. Computing it across the full panel including April would have quietly pulled label-period data into every single row's benchmark, which is a mistake I nearly made when I first wrote the `groupby`.

The cell below asserts these rather than trusting the paragraph.

In [11]:
# Does thin position coverage explain the top of the queue, and should I filter on it?
coverage_band = pd.cut(frame["position_coverage"], [0, 0.5, 0.9, 0.999, 1.0],
                       labels=["under 50%", "50-90%", "90-99%", "100%"])
gc = frame.groupby(coverage_band, observed=True)
print(f"decline rate by how much of a page's traffic carries a position   (base {BASE_RATE:.4f})")
print(pd.DataFrame({"n": gc.size(), "decline_rate": gc["y_declined"].mean()}).round(3).to_string())
print()
print(f"top 10  rows with under 50% coverage: {int((queue.head(10)['position_coverage'] < 0.5).sum())} / 10")
print(f"top 200 rows with under 50% coverage: {int((queue.head(200)['position_coverage'] < 0.5).sum())} / 200")
print()
print("would a coverage guard help? testing it rather than assuming:")
for minimum in (0.0, 0.5, 0.8, 0.9):
    guarded = frame[frame["position_coverage"] >= minimum].sort_values("score", ascending=False)
    print(f"  coverage >= {minimum:<4} n={len(guarded):>7,}  "
          f"p@10={guarded.head(10)['y_declined'].mean():.3f}  "
          f"p@50={guarded.head(50)['y_declined'].mean():.3f}  "
          f"p@200={guarded.head(200)['y_declined'].mean():.3f}")
print()
print("the guard makes it worse at every K, so I keep the rule and surface coverage in the CSV instead.")

decline rate by how much of a page's traffic carries a position   (base 0.5185)
                       n  decline_rate
position_coverage                     
under 50%           2936         0.678
50-90%             15142         0.620
90-99%             28226         0.566
100%               69778         0.471

top 10  rows with under 50% coverage: 3 / 10
top 200 rows with under 50% coverage: 7 / 200

would a coverage guard help? testing it rather than assuming:
  coverage >= 0.0  n=116,082  p@10=0.700  p@50=0.660  p@200=0.650


  coverage >= 0.5  n=113,185  p@10=0.500  p@50=0.640  p@200=0.645


  coverage >= 0.8  n=105,466  p@10=0.600  p@50=0.640  p@200=0.645
  coverage >= 0.9  n= 98,081  p@10=0.600  p@50=0.640  p@200=0.640

the guard makes it worse at every K, so I keep the rule and surface coverage in the CSV instead.


In [12]:
# Assert the leakage claims instead of just stating them.
SCORE_INPUTS = ["ctr_deficit", "imp_m0"]
FORBIDDEN = ["imp_next30", "y_declined"]

written = pd.read_csv(CSV_PATH, nrows=5)
leaked_columns = [c for c in FORBIDDEN if c in written.columns]

assert not leaked_columns, f"label leaked into the CSV: {leaked_columns}"
assert all(c in frame.columns for c in SCORE_INPUTS)

# Rebuild the score from scratch and confirm it matches: proves nothing else crept in.
rebuilt = (frame["ctr_deficit"] * np.log1p(frame["imp_m0"])).round(4)
assert np.allclose(rebuilt, frame["score"]), "score is not reproducible from its two stated inputs"

print("CSV columns:", list(written.columns))
print()
print(f"label columns in the CSV:            {leaked_columns or 'none'}")
print(f"score rebuilt from its 2 inputs:     matches exactly")
print(f"peer benchmark computed from:        March only, {int(frame['imp_m0'].sum()):,} impressions")
print()
print("no future-window inputs, no label-derived inputs, no product flags.")

CSV columns: ['rank', 'client_hash_id', 'content_hash_id', 'imp_m0', 'clk_m0', 'position', 'position_coverage', 'ctr', 'peer_ctr', 'ctr_deficit', 'score', 'action', 'reason_code']

label columns in the CSV:            none
score rebuilt from its 2 inputs:     matches exactly
peer benchmark computed from:        March only, 279,836,076 impressions

no future-window inputs, no label-derived inputs, no product flags.


## What this baseline is worth, and what Week 5 has to beat

**The frozen number: precision@50 = 0.660 against a 0.518 base rate, a 1.27x lift.** It holds up across K rather than wobbling, which the ML-03 staleness rule did not. That is the bar.

I am freezing it here, as the skill instructs. If the model beats it I will say by how much on this same slice with this same label, and if it does not, I will say that instead and recommend the rule, because an editor can read this rule and cannot read a random forest.

Three things I am carrying into ML-08:

1. **The impression-weighted recall is tiny.** The top 200 rows catch under 4% of the impressions that are actually at risk. The queue is right about which pages decline and nearly irrelevant to how much traffic is protected. Those are different objectives and I have only been optimising one.
2. **The eligibility floor question is finally settled.** I carried "the 50-impression floor is chosen, not derived" through ML-03 and ML-04. I tested it at 50, 100, 300 and 1,000 and the top of the queue does not move, because any page scoring highly is far above every one of those floors. It was a real question and the answer is that it does not matter for this rule. It would matter for a rate-based rule, which is worth remembering if the model changes shape.
3. **Ranking by a pure rate does not work here.** I tried ranking by the deficit ratio alone. It scored 0.420, below the base rate, because 50,346 pages have zero clicks and therefore tie at a deficit of exactly 1.0. That is the same tie-density trap that broke the staleness rule in ML-03, hit twice now, in the same project, from a different direction.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.